# Meta-Brain V2 — cérebro completo jogando Super Mario Bros

Este notebook é autocontido: não importa scripts do repositório, não exige clone local e baixa automaticamente os três arquivos oficiais do MaleCNS v1.0 quando eles ainda não existem. Ao final, salva vídeo pré/pós-treino, gráficos, CSV, JSON e checkpoint dentro de `/content/meta_brain_full_mario_v2`.

O guard de escala interrompe a execução se o arquivo carregado não tiver pelo menos 100.000 neurônios e 5 milhões de arestas. O rótulo `DEMONSTRATED_SUCCESS` só aparece se todos os episódios de avaliação pós-treino alcançarem a bandeira sem morrer.

In [ ]:
!pip -q install -U pandas pyarrow pillow opencv-python matplotlib gym-super-mario-bros nes-py requests

## Dados oficiais e parâmetros

O edgelist completo tem aproximadamente 1 GB e pode levar alguns minutos para baixar. Os URLs são os publicados para o release MaleCNS v1.0. Em uma nova execução, os arquivos já baixados são reutilizados.

In [ ]:
from pathlib import Path
import os

DATA_DIR = Path('/content/malecns_v1_0')
OUT_DIR = Path('/content/meta_brain_full_mario_v2')
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
FILES = {
    'annotations': 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/body-annotations-male-cns-v1.0-minconf-0.5.feather',
    'neurotransmitters': 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/body-neurotransmitters-male-cns-v1.0.feather',
    'weights': 'https://storage.googleapis.com/flyem-male-cns/v1.0/connectome-data/flat-connectome/connectome-weights-male-cns-v1.0-minconf-0.5.feather',
}
import requests

def download_if_missing(url):
    path = DATA_DIR / url.rsplit('/', 1)[-1]
    if path.exists() and path.stat().st_size > 1000:
        print('reusing', path.name, f'({path.stat().st_size/1e9:.2f} GB)')
        return path
    print('downloading', path.name)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8*1024*1024):
                if chunk: f.write(chunk)
    return path

paths = {k: download_if_missing(v) for k, v in FILES.items()}
print('data ready:', paths)

## Implementação completa

A conectividade é um reservatório recorrente esparso congelado. A única parte otimizada por RL é o readout motor; assim o notebook consegue executar o cérebro completo sem tentar materializar uma matriz densa de 166 mil × 166 mil. A entrada é visual: imagem 16×16 em tons de cinza mais diferença entre frames.

In [ ]:
import json, math, random, time, cv2
from PIL import Image
import numpy as np, pandas as pd, torch
import torch.nn as nn

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('Este experimento exige uma GPU CUDA no Colab; não faça uma execução CPU enganosa.')
DEVICE = torch.device('cuda')
MIN_NEURONS, MIN_EDGES = 100_000, 5_000_000

def col(df, names, default=None):
    for name in names:
        if name in df.columns: return name
    return default

ann = pd.read_feather(paths['annotations'])
nt_df = pd.read_feather(paths['neurotransmitters'])
edges = pd.read_feather(paths['weights'])
id_col = col(ann, ['bodyId', 'body_id'])
pre_col, post_col = col(edges, ['bodyId_pre', 'pre']), col(edges, ['bodyId_post', 'post'])
weight_col = col(edges, ['weight', 'synapse_count'], None)
if id_col is None or pre_col is None or post_col is None:
    raise ValueError('Schema MaleCNS não reconhecido.')
if weight_col is None: edges['weight'] = 1.0; weight_col = 'weight'
if len(ann) < MIN_NEURONS or len(edges) < MIN_EDGES:
    raise RuntimeError(f'Guard de cérebro completo falhou: {len(ann):,} neurônios e {len(edges):,} arestas.')

ids = ann[id_col].astype(np.int64).to_numpy()
idx = {int(x): i for i, x in enumerate(ids.tolist())}
pre_raw, post_raw = edges[pre_col].astype(np.int64).to_numpy(), edges[post_col].astype(np.int64).to_numpy()
keep = np.array([(int(a) in idx and int(b) in idx) for a,b in zip(pre_raw, post_raw)], dtype=bool)
pre = np.array([idx[int(x)] for x in pre_raw[keep]], dtype=np.int64)
post = np.array([idx[int(x)] for x in post_raw[keep]], dtype=np.int64)
ew = np.log1p(np.maximum(edges.loc[keep, weight_col].astype(np.float32).to_numpy(), 0))
indeg = np.zeros(len(ids), dtype=np.float32); np.add.at(indeg, post, ew*ew)
ew = ew / np.sqrt(np.maximum(indeg[post], 1e-6))

super_col = col(ann, ['superclass', 'super_class'], None)
if super_col is None: raise ValueError('Metadata não contém superclass.')
superclass = ann[super_col].fillna('').astype(str).str.lower()
sensory = np.flatnonzero(superclass.isin({'ol_sensory','cb_sensory','vnc_sensory','sensory_ascending','olsn'}).to_numpy())
motor = np.flatnonzero(superclass.isin({'descending_neuron','vnc_motor','cb_motor','dn','motor'}).to_numpy())
if len(sensory) == 0 or len(motor) == 0: raise ValueError('Não encontrei neurônios sensoriais/motores.')
print(f'FULL CONNECTOME VERIFIED: N={len(ids):,} E={len(pre):,} sensory={len(sensory):,} motor={len(motor):,}')

class Brain(nn.Module):
    def __init__(self, n, pre, post, ew, sensory, motor, actions, seed=123):
        super().__init__(); self.n=n
        self.pre=torch.tensor(pre,device=DEVICE); self.post=torch.tensor(post,device=DEVICE)
        self.ew=torch.tensor(ew,device=DEVICE); self.sens=torch.tensor(sensory,device=DEVICE); self.motor=torch.tensor(motor,device=DEVICE)
        gen=torch.Generator().manual_seed(seed)
        self.register_buffer('input_proj', torch.randn(512,len(sensory),generator=gen)/math.sqrt(512))
        self.W=nn.Parameter(torch.randn(actions,len(motor),generator=gen)*0.03); self.b=nn.Parameter(torch.zeros(actions))
    def step(self, code, state=None):
        with torch.no_grad():
            if code.ndim==1: code=code[None,:]
            if state is None: state=torch.zeros(code.shape[0],self.n,device=DEVICE)
            drive=torch.zeros_like(state); drive.index_add_(1,self.sens,code@self.input_proj)
            msg=torch.zeros_like(state); msg.index_add_(1,self.post,state[:,self.pre]*self.ew[None,:])
            state=0.90*state+0.10*torch.tanh(1.6*msg+drive)
        return state[:,self.motor]@self.W.t()+self.b, state

def reset_env(env, seed):
    try: out=env.reset(seed=seed)
    except TypeError: out=env.reset()
    return out[0] if isinstance(out,tuple) else out

def step_env(env, action):
    out=env.step(int(action))
    if len(out)==5:
        o,r,t,tr,i=out; return o,float(r),bool(t or tr),dict(i)
    o,r,d,i=out; return o,float(r),bool(d),dict(i)

def encode(frame, previous=None):
    a=np.asarray(frame)
    if a.ndim==3: a=a[...,:3].mean(-1)
    a=np.asarray(Image.fromarray(a.astype(np.uint8)).resize((16,16)),dtype=np.float32)/127.5-1
    d=a if previous is None else a-previous
    return np.concatenate([a.ravel(),d.ravel()]).astype(np.float32),a

import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace
from gym_super_mario_bros.actions import RIGHT_ONLY
env=JoypadSpace(gym_super_mario_bros.make('SuperMarioBros-1-1-v0'), RIGHT_ONLY)
brain=Brain(len(ids),pre,post,ew,sensory,motor,len(RIGHT_ONLY)).to(DEVICE)
print('actions:', len(RIGHT_ONLY), 'device:', DEVICE)

## Treino, avaliação e gravação

Aumente `ITERATIONS` se a curva ainda não estabilizar. O valor padrão é uma primeira rodada de 300 episódios; o notebook não transforma uma rodada curta em alegação de sucesso.

In [ ]:
ITERATIONS = int(os.environ.get('MB_MARIO_ITERS','300'))
EVAL_EPISODES, MAX_STEPS = 5, 2500

def rollout(greedy, seed, record=False):
    obs=reset_env(env,seed); state=None; previous=None; frames=[]; lps=[]; ents=[]; rewards=[]; info={}; total=0
    for _ in range(MAX_STEPS):
        if record: frames.append(np.asarray(obs).copy())
        code,previous=encode(obs,previous); logits,state=brain.step(torch.from_numpy(code).to(DEVICE),state)
        dist=torch.distributions.Categorical(logits=logits); action=int(logits.argmax(-1).item()) if greedy else int(dist.sample().item())
        if not greedy: lps.append(dist.log_prob(torch.tensor(action,device=DEVICE))); ents.append(dist.entropy())
        obs,r,done,info=step_env(env,action); rewards.append(r); total+=r
        if done: break
    success=bool(info.get('flag_get',False) or info.get('level_complete',False))
    return {'reward':float(total),'steps':len(rewards),'success':success,'died':bool(done and not success),'x_pos':float(info.get('x_pos',np.nan)),'flag_get':bool(info.get('flag_get',False)),'rewards':rewards,'lps':lps,'ents':ents,'frames':frames}

pre=[]
for i in range(EVAL_EPISODES): pre.append(rollout(True,10000+i,True))
opt=torch.optim.Adam([brain.W,brain.b],lr=2e-3); history=[]
for it in range(ITERATIONS):
    ep=rollout(False,20000+it); R=[]; running=0.0
    for r in ep['rewards'][::-1]: running=float(r)+0.99*running; R.append(running)
    ret=torch.tensor(R[::-1],device=DEVICE); ret=(ret-ret.mean())/(ret.std()+1e-6) if len(ret)>1 else ret
    loss=-(torch.stack(ep['lps'])*ret).mean()-0.01*torch.stack(ep['ents']).mean()
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_([brain.W,brain.b],1.0); opt.step()
    history.append({'iteration':it,'reward':ep['reward'],'loss':float(loss.detach().cpu()),'steps':ep['steps'],'x_pos':ep['x_pos'],'success':float(ep['success'])})
    if it==0 or (it+1)%10==0: print(f"iter={it:04d} reward={ep['reward']:+.1f} x={ep['x_pos']:.0f} steps={ep['steps']}")
post=[]
for i in range(EVAL_EPISODES): post.append(rollout(True,30000+i,True))

def summary(rows):
    return {'episodes':[{k:r[k] for k in ['reward','steps','success','died','x_pos','flag_get']} for r in rows], 'reward_mean':float(np.mean([r['reward'] for r in rows])), 'success_rate':float(np.mean([r['success'] for r in rows])), 'death_rate':float(np.mean([r['died'] for r in rows])), 'all_success_no_death':bool(all(r['success'] and not r['died'] for r in rows))}
pre_s,post_s=summary(pre),summary(post)
print('PRE:',pre_s); print('POST:',post_s)

## Artefatos finais

O JSON é o arquivo de controle da evidência. O campo `claim_status` separa uma execução concluída de uma demonstração real de sucesso.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Video, Image

def video(frames,path,fps=30):
    if not frames: return
    h,w=frames[0].shape[:2]; out=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),fps,(w,h))
    for f in frames:
        if f.ndim==2: f=np.repeat(f[...,None],3,-1)
        out.write(cv2.cvtColor(f.astype(np.uint8),cv2.COLOR_RGB2BGR))
    out.release()

def best_key(row):
    # Prefer a clean victory, then the farthest progress, then reward.
    return (int(row['success'] and not row['died']), float(np.nan_to_num(row['x_pos'], nan=-1e9)), row['reward'])

best_pre_i=max(range(len(pre)),key=lambda i:best_key(pre[i]))
best_post_i=max(range(len(post)),key=lambda i:best_key(post[i]))
pre_frames=video(pre[best_pre_i]['frames'],OUT_DIR/'full_brain_mario_v2_pre.mp4') or len(pre[best_pre_i]['frames'])
post_frames=video(post[best_post_i]['frames'],OUT_DIR/'full_brain_mario_v2_post.mp4') or len(post[best_post_i]['frames'])
h=pd.DataFrame(history); fig,ax=plt.subplots(1,3,figsize=(15,4))
ax[0].plot(h.reward,alpha=.35); ax[0].plot(h.reward.rolling(20,min_periods=1).mean()); ax[0].set_title('Reward')
ax[1].plot(h.x_pos); ax[1].set_title('X progress'); ax[2].plot(h.loss); ax[2].set_title('Policy loss')
fig.tight_layout(); fig.savefig(OUT_DIR/'full_brain_mario_v2_curves.png',dpi=160); plt.show()
h.to_csv(OUT_DIR/'full_brain_mario_v2_training.csv',index=False)
torch.save({'W_out':brain.W.detach().cpu(),'b_out':brain.b.detach().cpu(),'neurons':len(ids),'edges':int(keep.sum()),'motor':len(motor)},OUT_DIR/'full_brain_mario_v2_readout.pt')
result={'experiment':'Meta-Brain full-connectome Mario V2','full_connectome_guard':True,'graph':{'neurons':len(ids),'edges':int(keep.sum()),'sensory':len(sensory),'motor':len(motor)},'config':{'iterations':ITERATIONS,'eval_episodes':EVAL_EPISODES,'max_steps':MAX_STEPS,'device':str(DEVICE),'observation':'16x16 grayscale + frame difference','trainable':'motor readout only'},'best_video_episode':{'pre':best_pre_i+1,'post':best_post_i+1,'post_metrics':{k:post[best_post_i][k] for k in ['reward','steps','success','died','x_pos','flag_get']}},'video_frames':{'pre':pre_frames,'post':post_frames},'pre_eval':pre_s,'post_eval':post_s,'claim_status':'DEMONSTRATED_SUCCESS' if post_s['all_success_no_death'] else 'RUN_COMPLETED_BUT_SUCCESS_NOT_DEMONSTRATED'}
(OUT_DIR/'full_brain_mario_v2_results.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print(json.dumps(result,indent=2)); print('outputs:',*[p for p in sorted(OUT_DIR.iterdir())],sep='\n')
display(Image(filename=str(OUT_DIR/'full_brain_mario_v2_curves.png')))
display(Video(filename=str(OUT_DIR/'full_brain_mario_v2_pre.mp4'),embed=True))
display(Video(filename=str(OUT_DIR/'full_brain_mario_v2_post.mp4'),embed=True))